# CattleVision-360 — Full Retraining Pipeline (Google Colab)

This notebook:
1. Downloads the Beef Cattle Muzzle dataset from Kaggle
2. Installs all dependencies
3. Trains YOLOv8s for 100 epochs (muzzle detection)
4. Trains ResNet18 + ArcFace for 50 epochs (biometric embeddings)
5. Builds FAISS vector index
6. Downloads all trained weights as a zip file

**Runtime**: Set to GPU (Runtime → Change runtime type → GPU → T4)

In [ ]:
# CELL 1: Install dependencies
!pip install -q ultralytics faiss-cpu torchvision opencv-python-headless pyyaml

In [ ]:
# CELL 2: Setup Kaggle & Download Dataset
import os
from google.colab import userdata

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Using Kaggle credentials from Colab Secrets")
except:
    os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME_HERE"
    os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY_HERE"
    print("Using manual Kaggle credentials - REPLACE WITH YOURS!")

print("Downloading dataset from Kaggle...")
!kaggle datasets download parthibanmarimuthu/beefcattle-muzzle -p ./dataset --unzip
print("Dataset downloaded and extracted!")

In [ ]:
# CELL 3: Verify Dataset Structure
import os
from pathlib import Path

dataset_path = Path('./dataset')
cows = [d.name for d in dataset_path.iterdir() if d.is_dir()]
print(f"Found {len(cows)} cattle identities")
print(f"First 10: {cows[:10]}")

total_imgs = sum(len(list(d.glob('*.jpg'))) for d in dataset_path.iterdir() if d.is_dir())
print(f"Total images: {total_imgs}")

In [ ]:
# CELL 4: Create retrain.py script (FIXED version)
retrain_code = '''"""
CattleVision-360 Full Retraining Pipeline - FIXED
"""
import argparse, json, os, random, shutil
from pathlib import Path
import cv2, faiss, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from ultralytics import YOLO
import yaml

class ArcFace(nn.Module):
    def __init__(self, in_features, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s, self.m = s, m
        self.cos_m, self.sin_m = np.cos(m), np.sin(m)
        self.th = np.cos(np.pi - m)
        self.mm = np.sin(np.pi - m) * m
        self.class_num = num_classes
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.normal_(self.weight, mean=0.0, std=0.01)

    def forward(self, features, labels):
        cos_theta = F.linear(F.normalize(features), F.normalize(self.weight))
        sin_theta = torch.sqrt(1.0 - torch.clamp(cos_theta * cos_theta, 0, 1))
        cos_theta_m = cos_theta * self.cos_m - sin_theta * self.sin_m
        one_hot = F.one_hot(labels, self.class_num).float()
        cos_theta_m = torch.where(one_hot.bool(), cos_theta_m, cos_theta)
        cos_theta_m = torch.where(cos_theta < self.th, cos_theta - self.mm, cos_theta_m)
        return self.s * cos_theta_m

def apply_clahe(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)

def build_yolo_dataset(src_root, out_root, train_frac=0.8, auto_box_frac=0.9):
    out_root.mkdir(parents=True, exist_ok=True)
    for split_name in ["train", "val"]:
        (out_root / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (out_root / "labels" / split_name).mkdir(parents=True, exist_ok=True)
    
    cows = sorted([d for d in src_root.iterdir() if d.is_dir()])
    records = []
    for cow_dir in cows:
        for img_path in list(cow_dir.glob("*.jpg")) + list(cow_dir.glob("*.png")):
            records.append((cow_dir.name, img_path))
    
    random.shuffle(records)
    n_train = int(len(records) * train_frac)
    split = {"train": records[:n_train], "val": records[n_train:]}
    
    for split_name, items in split.items():
        for cow_name, img_path in items:
            new_img = out_root / "images" / split_name / img_path.name
            new_lbl = out_root / "labels" / split_name / (img_path.stem + ".txt")
            shutil.copy2(img_path, new_img)
            with open(new_lbl, "w") as f:
                f.write(f"0 0.5 0.5 {auto_box_frac} {auto_box_frac}\\n")
    
    cfg = {"path": str(out_root.resolve()), "train": "images/train", "val": "images/val", "nc": 1, "names": {0: "muzzle"}}
    (out_root / "data.yaml").write_text(yaml.safe_dump(cfg))
    return out_root / "data.yaml"

class MuzzleEmbeddingDataset(Dataset):
    def __init__(self, identities_root, transform):
        self.samples, self.labels, self.class_to_idx = [], [], {}
        self.transform = transform  # FIXED: Save transform attribute!
        for cow_dir in sorted(identities_root.iterdir()):
            if not cow_dir.is_dir():
                continue
            if cow_dir.name not in self.class_to_idx:
                self.class_to_idx[cow_dir.name] = len(self.class_to_idx)
            for img in list(cow_dir.glob("*.jpg")) + list(cow_dir.glob("*.png")):
                self.samples.append(str(img))
                self.labels.append(self.class_to_idx[cow_dir.name])
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, i):
        img = np.array(Image.open(self.samples[i]).convert("RGB"))
        img = apply_clahe(img)
        img = self.transform(Image.fromarray(img))
        return img, int(self.labels[i])

def train_yolo(data_yaml, epochs, imgsz, batch, device, out_dir):
    print(f"Training YOLO for {epochs} epochs...")
    model = YOLO("yolov8s.pt")
    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        patience=30,
        cos_lr=True,
        augment=True,
        project=str(out_dir / "runs"),
        name="yolo",
        exist_ok=True
    )
    best = out_dir / "runs" / "yolo" / "weights" / "best.pt"
    try:
        metrics = model.metrics
        info = {
            "yolo_model": "yolov8s",
            "epochs": epochs,
            "map50": float(metrics.box.map50) if metrics.box else None,
            "map50_95": float(metrics.box.map) if metrics.box else None
        }
        (out_dir / "model_info.json").write_text(json.dumps(info, indent=2))
        print(f"YOLO mAP50={info['map50']:.4f}, mAP50-95={info['map50_95']:.4f}")
    except Exception as e:
        print(f"Could not save metrics: {e}")
    return best

def train_embedder(identities_root, epochs, batch, device, out_dir):
    print(f"Training ResNet18+ArcFace for {epochs} epochs...")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    ds = MuzzleEmbeddingDataset(identities_root, transform=transform)
    loader = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=True)
    num_classes = max(ds.labels) + 1
    
    backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    backbone.fc = nn.Identity()
    embedding_net = nn.Sequential(backbone, nn.BatchNorm1d(512), nn.Dropout(0.4), nn.Linear(512, 512)).to(device)
    arc_head = ArcFace(512, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW([
        {"params": embedding_net.parameters(), "lr": 1e-4},
        {"params": arc_head.parameters(), "lr": 5e-3}
    ])
    
    for epoch in range(epochs):
        running = 0.0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            feats = F.normalize(embedding_net(imgs), dim=1)
            loss = criterion(arc_head(feats, labels), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {running/len(loader):.4f}")
    
    out_path = out_dir / "resnet_arcface.pth"
    torch.save(embedding_net.state_dict(), out_path)
    return out_path

def rebuild_faiss(identities_root, embedder_path, out_dir, faiss_dim=512):
    print("Building FAISS index...")
    tfm = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    infer_ds = MuzzleEmbeddingDataset(identities_root, tfm)
    loader = DataLoader(infer_ds, batch_size=64, shuffle=False)
    idx_to_name = {v: k for k, v in infer_ds.class_to_idx.items()}
    
    backbone = resnet18(weights=None)
    backbone.fc = nn.Identity()
    full_net = nn.Sequential(backbone, nn.BatchNorm1d(512), nn.Dropout(0.4), nn.Linear(512, faiss_dim))
    full_net.load_state_dict(torch.load(embedder_path, map_location="cpu"))
    full_net.eval()
    
    index = faiss.IndexFlatIP(faiss_dim)
    cow_ids = []
    with torch.no_grad():
        for imgs, labels in loader:
            vecs = F.normalize(full_net(imgs), p=2, dim=1).numpy().astype("float32")
            index.add(vecs)
            cow_ids.extend([idx_to_name[int(l)] for l in labels.numpy()])
    
    faiss.write_index(index, str(out_dir / "cattle_muzzle_faiss.bin"))
    (out_dir / "label_map.json").write_text(json.dumps(cow_ids))
    print(f"FAISS index built with {index.ntotal} vectors")

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    out = Path("./artifacts")
    out.mkdir(parents=True, exist_ok=True)
    data_root = Path("./dataset")
    
    yolo_yaml = build_yolo_dataset(data_root, out / "yolo_dataset")
    train_yolo(yolo_yaml, epochs=100, imgsz=224, batch=32, device=device, out_dir=out)
    emb_path = train_embedder(data_root, epochs=50, batch=32, device=device, out_dir=out)
    rebuild_faiss(data_root, emb_path, out)
    
    print("\\n=== RETRAINING COMPLETE ===")
    print(f"Artifacts saved to: {out}")
'''

with open('retrain.py', 'w') as f:
    f.write(retrain_code)
print("retrain.py script created (FIXED version)!")

In [ ]:
# CELL 5: Run the Full Retraining Pipeline
!python retrain.py

In [ ]:
# CELL 6: Download Trained Models
from google.colab import files
import shutil

shutil.make_archive('trained_artifacts', 'zip', './artifacts')
print("Downloading trained_artifacts.zip...")
files.download('trained_artifacts.zip')